# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template and walkthrough for loading and exploring the [FAIR² dataset package](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, which ensures that record set and field identifiers (`@id`) are clearly and uniquely referenced throughout.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print name and description
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets (`@id`s), their fields, and associated column `@id`s. We show their identifiers below as retrieved via the schema, which can be referenced in subsequent code blocks.

In [ ]:
# List all record sets and their fields by `@id`
record_sets = []
for rset in dataset.record_sets:
    print(f"RecordSet @id: {rset.id}")
    record_sets.append(rset.id)
    for field in rset.fields:
        print(f"  Field @id: {field.id} | name: {field.name} | dataType: {field.data_type} | columns: {[col.id for col in field.columns]}")
    print('-' * 60)
# Save record_sets for later use
print(f"All record sets: {record_sets}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references are to their Croissant `@id` identifiers as shown above.

In [ ]:
# Create DataFrames for each record set by @id
dataframes = dict()
for record_set_id in record_sets:
    print(f"Loading records for RecordSet: {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
You can now select a specific record set and numeric field for further analysis. Below, we select the main record set and numeric field (using their `@id`s).

Common EDA steps include filtering, normalization, and grouping by attributes. Update the variables as needed based on the data overview above.

In [ ]:
#### Set the relevant record set and field @ids from the overview output above
# For this dataset, the main record set usually contains protein entries (e.g. 'cr:RecordSet_0'),
# and likely has numeric columns such as 'cr:abundance', 'cr:coverage', etc.
if record_sets:
    rec_id = record_sets[0]  # Update if you want a different one
    df = dataframes.get(rec_id)
    if df is not None and not df.empty:
        print(f"First five columns: {df.columns[:5].tolist()}")
        # Try to find a likely numeric field, otherwise show all
        potential_numeric = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
        if not potential_numeric:
            sample = df.iloc[0].to_dict()
            for k, v in sample.items():
                try:
                    float(v)
                    potential_numeric.append(k)
                except:
                    continue
        
        if potential_numeric:
            numeric_field = potential_numeric[0]
            print(f"Using numeric field: {numeric_field}")
            threshold = df[numeric_field].astype(float).mean()  # use mean as threshold
            filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.2f} (total: {len(filtered_df)}):")
            print(filtered_df.head())
            # Normalize the field
            filtered_df = filtered_df.copy()
            filtered_df[f"{numeric_field}_normalized"] = (
                pd.to_numeric(filtered_df[numeric_field], errors='coerce') - filtered_df[numeric_field].astype(float).mean()
            ) / filtered_df[numeric_field].astype(float).std()
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
            # Find a likely grouping field (categorical)
            non_numeric = [col for col in df.columns if col not in potential_numeric]
            group_field = None
            for col in non_numeric:
                # Use as group if <50 unique values, not null, not empty
                nunique = df[col].nunique(dropna=True) if col in df else 0
                if 1 < nunique < 50:
                    group_field = col
                    break
            if group_field:
                print(f"Grouping by: {group_field}")
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(grouped_df.head())
        else:
            print("No numeric fields detected.")
    else:
        print("No records available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field using a histogram, and compare means across different protein groups if relevant fields exist.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped_df.index, y=grouped_df.values)
        plt.title(f"Mean {numeric_field} per {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- We demonstrated step-by-step data extraction from a FAIR² Croissant-defined dataset using unique `@id` references for record sets and fields.
- Sample data was loaded into pandas DataFrames, filtered by a numeric field, normalized, optionally grouped by a categorical field, and visualized.
- The Croissant metadata and structure facilitates robust, self-describing analytics workflows for proteomics and clinical scientific datasets.

You can continue by applying domain-specific analytics, ML modeling, or further visualizations.